# 09. Context i State: dwa różne źródła danych

W lekcjach 06 i 07 poznaliśmy `state` i `context` osobno. Teraz prześledzimy **ten sam przebieg** i sprawdzimy, skąd każda wartość pochodzi, kto może ją odczytać i co dzieje się po następnym wywołaniu. Przykład używa jawnie zaprogramowanych odpowiedzi zamiast prawdziwego LLM, więc wynik jest powtarzalny. Nie symulujemy logowania: kontekst traktujemy jak dane, które aplikacja wcześniej sprawdziła.

[Dokumentacja LangChain: źródła danych i context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering) · [LangGraph: state i runtime context](https://docs.langchain.com/oss/python/langgraph/graph-api)

In [ ]:
# Jedyna komórka z importami.
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import NotRequired, TypedDict

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.faults import ScriptedModel


## 1. Najpierw zwykły Python

`state` to dane, które **zmieniają się podczas pracy**: historia wiadomości, rezultat narzędzia, licznik, status zadania. `context` to dane, które **aplikacja dostarcza do konkretnego wywołania**: ustalony identyfikator użytkownika, język, uprawnienia lub uchwyt do usługi. Samo przekazanie kontekstu nie dopisuje go do promptu ani stanu.

Poniższe słowniki są tylko obrazkiem. Właściwe API pojawi się dalej.

In [ ]:
state = {"messages": ["Jaki mam plan?"], "turns": 0}
context = {"viewer_id": "u-101", "language": "pl"}
state["turns"] += 1
state, context


| Pytanie | State | Runtime context |
|---|---|---|
| Kto podaje wartość? | wejście użytkownika, model, narzędzie lub kod grafu | kod aplikacji przy `invoke(..., context=...)` |
| Czy może zmieniać się podczas przebiegu? | tak, przez zwracane aktualizacje | traktuj jako wejście tylko do odczytu |
| Czy checkpointer może zachować ją dla `thread_id`? | tak | nie; dostarcz ponownie przy kolejnym wywołaniu |
| Czy model widzi ją automatycznie? | tylko wiadomości i pola jawnie włączone do żądania | nie; dopiero gdy kod umieści wartość w promptcie lub wyniku narzędzia |
| Jak ją odczytać? | `result["messages"]`, `runtime.state`, argument `state` hooka/węzła | `runtime.context` |

**Granica zaufania:** `viewer_id` i uprawnienia bierze serwer z wcześniej sprawdzonej sesji. Tekst użytkownika ani `state` wygenerowany przez model nie powinny nadawać uprawnień. Checkpointer przechowuje stan wątku, a nie autoryzuje żądań.

## 2. Kontekst podaje aplikacja, narzędzie odczytuje stan

`RequestContext` jest małym, niemutowalnym kontraktem. Słownik profili zastępuje bazę. Model zna tylko nazwę narzędzia; `viewer_id` nie jest argumentem narzędzia, który mógłby wymyślić model. `runtime.state` w narzędziu pokazuje aktualny stan agenta, a `runtime.context` dane bieżącego wywołania.

In [ ]:
@dataclass(frozen=True)
class RequestContext:
    viewer_id: str
    language: str


PROFILES = {"u-101": "Plus", "u-202": "Basic"}
observed = []


@tool
def my_plan(runtime: ToolRuntime[RequestContext]) -> str:
    """Odczytaj plan użytkownika ustalonego przez aplikację."""
    # ID pochodzi z context. Model nie przekazuje go jako argumentu narzędzia.
    viewer_id = runtime.context.viewer_id
    # State zawiera wiadomości tej rozmowy; nie jest źródłem tożsamości.
    observed.append({"viewer_id": viewer_id, "messages_seen": len(runtime.state["messages"])})
    return PROFILES[viewer_id]


## 3. Agent LangChain: wejście, narzędzie, wynik

Sztuczny model wykona zaplanowane wywołanie narzędzia i zwróci zaplanowaną odpowiedź. Nie ocenia sam, który plan jest poprawny. Poprawną wartość zwraca Python z `PROFILES`; w prawdziwej aplikacji byłby to odczyt z uprawnionej bazy. Używamy `context_schema` jako kontraktu dla `context=...`.

In [ ]:
scripted_model = ScriptedModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-a"}]),
    AIMessage(content="Plan odczytany z narzędzia."),
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-b"}]),
    AIMessage(content="Drugi plan odczytany z narzędzia."),
])
agent = create_agent(
    model=scripted_model,
    tools=[my_plan],
    context_schema=RequestContext,
    checkpointer=InMemorySaver(),
)
thread = {"configurable": {"thread_id": "lekcja-09"}}
first = agent.invoke(
    {"messages": [("user", "Jaki mam plan?")]},
    context=RequestContext(viewer_id="u-101", language="pl"),
    config=thread,
)
[(m.type, m.content) for m in first["messages"]], observed


`first["messages"]` jest stanem po przebiegu. Zawiera wiadomość użytkownika, prośbę modelu o narzędzie, wynik narzędzia i odpowiedź modelu. W odpowiedzi sztucznego modelu nie ma jeszcze nazwy planu: tekst jest zaprogramowany. To celowo oddziela **decyzję modelu** od **danych zwróconych przez Python**.

In [ ]:
second = agent.invoke(
    {"messages": [("user", "A teraz mój plan?")]},
    context=RequestContext(viewer_id="u-202", language="en"),
    config=thread,
)
[(m.type, m.content) for m in second["messages"][-4:]], observed[-1]


Ten sam `thread_id` zachował historię wiadomości, lecz **nie zamroził kontekstu**: druga tura dostała `u-202`. W aplikacji produkcyjnej nie zmieniaj w ten sposób właściciela cudzego wątku; tutaj to eksperyment pokazujący granicę API. Właściciela wątku trzeba sprawdzić poza grafem. `InMemorySaver` trzyma dane tylko przez życie tego obiektu w bieżącym procesie.

In [ ]:
saved = agent.get_state(thread)
len(saved.values["messages"]), [m.content for m in saved.values["messages"] if m.type == "tool"]


## 4. Delikatne spojrzenie pod spodem: LangGraph

Agent LangChain działa na grafie. Graf ma **węzły** wykonujące kod i **state**, które przechodzi między nimi. `StateGraph` opisuje kształt tego stanu. Węzeł dostaje bieżący stan oraz `Runtime` z kontekstem. Zwraca **częściową aktualizację**, a graf składa nowy stan. Tu jest tylko jeden węzeł; nie uczymy jeszcze trasowania ani reducerów.

In [ ]:
class VisitState(TypedDict):
    question: str
    turns: NotRequired[int]
    answer: NotRequired[str]


def answer_question(state: VisitState, runtime: Runtime[RequestContext]) -> dict:
    # State może zmieniać się między turami; context jest podany dla tej tury.
    turns = state.get("turns", 0) + 1
    plan = PROFILES[runtime.context.viewer_id]
    return {"turns": turns, "answer": f"Tura {turns}: plan {plan}. Pytanie: {state['question']}"}


builder = StateGraph(VisitState, context_schema=RequestContext)
builder.add_node("answer", answer_question)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile(checkpointer=InMemorySaver())
graph_thread = {"configurable": {"thread_id": "graf-09"}}


In [ ]:
graph_first = graph.invoke(
    {"question": "Jaki plan?"},
    context=RequestContext("u-101", "pl"),
    config=graph_thread,
)
graph_second = graph.invoke(
    {"question": "A teraz?"},
    context=RequestContext("u-202", "en"),
    config=graph_thread,
)
graph_first, graph_second


`turns` rośnie, bo graf odczytuje poprzedni zapisany state. `answer` i `question` są polami stanu. `viewer_id` służy węzłowi do obliczenia odpowiedzi, ale nie pojawia się jako pole stanu. Bez checkpointera każde wywołanie zaczęłoby od wejściowego stanu. Nowy `thread_id` też zacznie od początku.

In [ ]:
other_thread = graph.invoke(
    {"question": "Jaki plan?"},
    context=RequestContext("u-101", "pl"),
    config={"configurable": {"thread_id": "graf-inny"}},
)
other_thread


## 5. Jak wybierać poprawne miejsce na dane?

- **Wiadomości, postęp zadania, wynik narzędzia:** state. Aktualizuj przez zwrot z węzła/hooka albo `Command(update=...)` z narzędzia, nie przez przypadkową mutację `runtime.state`.
- **Tożsamość, zakres uprawnień, język bieżącego żądania, klient bazy:** runtime context. Ustalaj w aplikacji i podawaj przy każdym `invoke`. Nie wkładaj sekretów do wiadomości.
- **Instrukcje i fakty potrzebne modelowi teraz:** jawnie do promptu lub wyniku narzędzia. Samo `context=...` niczego modelowi nie pokazuje.
- **Wiedza z innych rozmów:** osobny trwały magazyn lub baza. Checkpointer stanu jednej rozmowy nie jest takim magazynem.

**Ćwiczenie:** dodaj do `RequestContext` pole `region`, a do `VisitState` pole `last_plan`. W węźle odczytaj pierwsze z `runtime.context`, drugie aktualizuj przez zwrócony słownik. Sprawdź dwa wątki i dwa różne konteksty dla tego samego wątku.